In [1]:
from pathlib import Path
from datetime import datetime
from typing import List
from types import SimpleNamespace
import pickle
import numpy as np
import torch
from pymatgen.core import Structure, Lattice, Species, Element


from chggen.pl_data.dataset import CHGNetDataset
from chggen.pl_modules.model import CHGGen
from chggen.common.data_utils import get_scaler_from_data_list
from chggen.pl_modules.decoder import NequipDecoder

from torch_geometric.data import Data
from torch import nn
import torch.nn.functional as F



def get_scaler(dataset, use_prop_scaler = False, 
               scaler_path = None):
    # Load once to compute property scaler
    if scaler_path is None:
        lattice_scaler = get_scaler_from_data_list(
            dataset.cached_data,
            key='scaled_lattice')
        if use_prop_scaler:
            NotImplementedError("Not implemented the multi prop scaler yet.")
    else:
        lattice_scaler = torch.load(
            Path(scaler_path) / 'lattice_scaler.pt')
    return lattice_scaler



/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chggen = CHGGen.load_from_checkpoint(checkpoint_path="./test_models/perov/epoch=9.ckpt")

CHGNet initialized with 400,438 parameters
CHGNet initialized with 400,438 parameters


/home/zhongpc/anaconda3/envs/cdvae/lib/python3.9/site-packages/torch/jit/_check.py:172: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn("The TorchScript type system doesn't support "


In [3]:
with open('./test_models/lattice_scaler_perov', 'rb') as fp:
    lattice_scaler = pickle.load(fp)
    
chggen.lattice_scaler = lattice_scaler

In [4]:
!nvidia-smi

Thu Sep 28 15:31:04 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100 80G...  On   | 00000000:01:00.0 Off |                    0 |
| N/A   47C    P0    79W / 300W |   6690MiB / 81920MiB |      1%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA A100 80G...  On   | 00000000:25:00.0 Off |                    0 |
| N/A   

In [5]:
chggen.to(torch.device('cuda:0'))

CHGGen(
  (encoder): CHGNet_encoder(
    (composition_model): AtomRef(
      (fc): Linear(in_features=94, out_features=1, bias=False)
    )
    (graph_converter): CrystalGraphConverter(algorithm='fast', atom_graph_cutoff=5, bond_graph_cutoff=3)
    (atom_embedding): AtomEmbedding(
      (embedding): Embedding(94, 64)
    )
    (bond_basis_expansion): BondEncoder(
      (rbf_expansion_ag): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
      (rbf_expansion_bg): RadialBessel(
        (smooth_cutoff): CutoffPolynomial()
      )
    )
    (bond_embedding): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_ag): Linear(in_features=9, out_features=64, bias=False)
    (bond_weights_bg): Linear(in_features=9, out_features=64, bias=False)
    (angle_basis_expansion): AngleEncoder(
      (fourier_expansion): Fourier()
    )
    (angle_embedding): Linear(in_features=9, out_features=64, bias=False)
    (atom_conv_layers): ModuleList(
      (0-3): 4 x AtomConv(


In [6]:
# langevin dynamics
ld_kwargs = SimpleNamespace(n_step_each = 10,
                            step_lr = 1e-4,
                            min_sigma = 0,
                            save_traj = True,
                            disable_bar = False,
                            compute_force = True,
                            beta_c = 0, 
                            beta_f = 1e-3)

z = torch.rand(3, 64, requires_grad= True, device = chggen.device)


results = chggen.langevin_dynamics_guidance(z = z, 
                                            prop_guidance = torch.tensor(1.0, device = chggen.device), 
#                                             gt_num_atoms= torch.tensor([12]), 
#                                             gt_atom_types= torch.tensor([3,3,3, 25,25,25, 8,8,8,8,8,8]), 
                                            ld_kwargs= ld_kwargs)


/home/zhongpc/chggen/chggen/common/data_utils.py:625: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)
  0%|                                                                                                                          | 0/50 [00:00<?, ?it/s]

InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
N2O2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F


  2%|██▎                                                                                                               | 1/50 [00:04<03:25,  4.20s/it]

InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F


  4%|████▌                                                                                                             | 2/50 [00:06<02:25,  3.03s/it]

...

InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F


 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████▋  | 49/50 [00:47<00:00,  1.05it/s]

InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:48<00:00,  1.02it/s]

InSnO2F
AlInO2F
RuNO2F
InSnO2F
AlInO2F
RuNO2F


In [7]:
# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
frac_coords = results['frac_coords']
atom_types = results['atom_types']

batch = torch.arange(len(num_atoms), device = chggen.device)
batch = batch.repeat_interleave(num_atoms)

for ii in range(len(num_atoms)):
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths[ii,0].cpu(), b = lengths[ii,1].cpu(), c = lengths[ii,2].cpu(),
                                   alpha= angles[ii, 0].cpu(), beta= angles[ii,1].cpu(), gamma=angles[ii, 2].cpu())
                                   
    frac_ = frac_coords[indices]
    type_ = atom_types[indices]
    species_ = [Element.from_Z(ele_Z) for ele_Z in type_]
    
    s_gen = Structure(lattice= Latt , species= species_, coords= frac_.cpu().detach().numpy(),
                      to_unit_cell=False,coords_are_cartesian=False);
    print(s_gen.composition)
    s_gen.to(filename= './test_models/structures/prop_guidance_' + str(ii) + '.cif')
    

0 tensor([0, 1, 2, 3, 4], device='cuda:0')
O2 F1 Sn1 In1
1 tensor([5, 6, 7, 8, 9], device='cuda:0')
O2 F1 Al1 In1
2 tensor([10, 11, 12, 13, 14], device='cuda:0')
N1 Ru1 O2 F1


In [8]:
!mkdir ./test_models/structures/perov_traj/

mkdir: cannot create directory ‘./test_models/structures/perov_traj/’: File exists


In [9]:


# save the results from langevin dynamics
lengths = results['lengths']
angles= results['angles']
num_atoms = results['num_atoms']
all_frac_coords = results['all_frac_coords']
all_atom_types = results['all_atom_types']

batch = torch.arange(len(num_atoms), device = chggen.device)
batch = batch.repeat_interleave(num_atoms)

for ii in range(len(num_atoms)):
    if ii > 0:
        break
    indices = torch.where(batch == ii)[0]
    print(ii, indices)
    if len(indices) == 0:
        continue
    
    
    Latt = Lattice.from_parameters(a = lengths[ii,0].cpu(), b = lengths[ii,1].cpu(), c = lengths[ii,2].cpu(),
                                   alpha= angles[ii, 0].cpu(), beta= angles[ii,1].cpu(), gamma=angles[ii, 2].cpu())
    
    for jj in range(0, len(all_frac_coords), 10):
        
                                   
        frac_ = all_frac_coords[jj, indices, :]
        type_ = all_atom_types[jj, indices]
        species_ = [Element.from_Z(ele_Z) for ele_Z in type_]

        s_gen = Structure(lattice= Latt , species= species_, coords= frac_.cpu().detach().numpy(),
                          to_unit_cell=False,coords_are_cartesian=False);
        print(s_gen.composition)
        s_gen.to(filename= './test_models/structures/perov_traj/traj_guidance_' + str(ii) + '_traj_' + str(jj) + '.cif')


0 tensor([0, 1, 2, 3, 4], device='cuda:0')
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
O2 F1 Sn1 In1
